# 7장 Advanced Text Generation Techniques and Tools — SageMaker AI Studio(JupyterLab)용

<!-- origin-notice -->
> **원본 노트북:** [chapter07/Chapter 7 - Advanced Text Generation Techniques and Tools.ipynb](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter07/Chapter%207%20-%20Advanced%20Text%20Generation%20Techniques%20and%20Tools.ipynb)  
> **원본 저장소:** [HandsOnLLM/Hands-On-Large-Language-Models](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models) (Apache License 2.0, © 2024 Jay Alammar & Maarten Grootendorst)  
> 이 노트북은 원본을 SageMaker AI Studio에서 실행되도록 수정하고 한국어 설명·보충 실험을 추가한 것입니다. 변경 내용은 노트북 안의 "책과 다를 수 있는 점"에 적혀 있습니다.

원본: HandsOnLLM/Hands-On-Large-Language-Models `chapter07` 노트북(Apache 2.0)을 Studio에서 돌아가도록 수정한 것입니다. 6장까지는 프롬프트 한 번으로 답을 받았다면, 7장은 LLM 바깥에 부품을 붙여 시스템으로 만듭니다. 부품을 잇는 도구로 LangChain을 씁니다.

| 절 | 하는 일 | 책 위치(PDF 쪽) |
|---|---|---|
| 1 | 양자화(quantization)된 GGUF 모델을 llama.cpp로 불러오기 (+ 비트 수와 정밀도) | 281–283 |
| 2 | 프롬프트 템플릿과 LLM을 **체인(chain)**으로 잇기, 체인 여러 개를 순서대로 연결 (+ 현재 권장 문법 LCEL) | 284–292 |
| 3 | 대화 **메모리(memory)** 3종: 전체 버퍼·최근 k턴 창·요약 (+ 메모리별 토큰 수 비교) | 293–305 |
| 4 | **에이전트(agent)**: ReAct 프롬프트로 검색·계산 도구 쓰기 (+ 실행 기록 보기, 직접 짠 ReAct 루프, 도구 호출 API) | 306–315 |

**권장 환경:** `ml.g4dn.xlarge`(T4 16GB), 스토리지 **50GB 이상**. 1·2장 캐시(약 9GB)에 이 장의 GGUF 파일(fp16 7.6GB)이 더해집니다. 스토리지가 30GB라면 1절에서 4비트 파일(2.4GB)을 고르세요.

`(+ …)` 표시는 책에 없는 보충 실험이고, **(책 밖)** 표시는 책 출간(2024) 이후 내용입니다.

> **버전 결정과 이유.** 책은 `langchain 0.2.x`를 쓰는데, 2025년 10월 LangChain 1.0에서 `LLMChain`, `ConversationBufferMemory`, `RetrievalQA`, `AgentExecutor` 같은 예전 클래스가 본 패키지에서 빠져 `langchain-classic`으로 옮겨졌습니다. 이 노트북은 **현재 버전 + `langchain-classic`** 조합을 씁니다. 책 코드는 import 경로만 바뀌고 나머지는 그대로이며, 각 절에 현재 권장 방식(LCEL)을 함께 붙였습니다. 옛 버전(0.2.x)으로 고정하지 않은 이유는 최신 transformers·pydantic과 충돌하기 때문입니다. 아래 버전 조합은 2026-11에 설치·실행을 확인했습니다.

## 0. 환경 점검과 패키지 설치

GPU·디스크를 확인하고 LangChain 계열을 지정 버전으로 설치합니다. Studio에 기본으로 들어 있는 Jupyter AI 확장이 예전 LangChain을 쓰고 있어 pip이 의존성 경고를 낼 수 있는데, 노트북 커널 동작에는 영향이 없습니다.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h ~ | tail -1
import importlib.metadata as md_, subprocess, sys
pins = {
    "langchain-core": "1.7.6", "langchain-classic": "1.0.8", "langchain-community": "0.4.2",
    "langchain-huggingface": "1.2.3", "ddgs": "9.16.1", "numexpr": None, "huggingface_hub": None,
}
todo = []
for pkg, ver in pins.items():
    try:
        have = md_.version(pkg)
        if ver and have != ver: todo.append(f"{pkg}=={ver}")
    except md_.PackageNotFoundError:
        todo.append(f"{pkg}=={ver}" if ver else pkg)
if todo:
    print("설치:", todo)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *todo])
    print("설치 완료 → 커널을 재시작(Kernel > Restart)한 뒤 이 셀부터 다시 실행하세요.")
else:
    print("추가 설치 불필요")

### llama-cpp-python 설치 (GPU용)

GGUF 파일을 돌리는 엔진은 llama.cpp이고, 파이썬 바인딩이 `llama-cpp-python`입니다. PyPI에는 소스만 있어서 그냥 설치하면 수십 분 동안 컴파일합니다. 그래서 **CUDA용으로 미리 빌드된 wheel**을 받습니다. 프로젝트가 제공하는 CUDA wheel은 2026-11 기준 0.3.19, Python 3.9–3.12까지입니다.

- 위 조건이 안 맞으면 소스 빌드(`CMAKE_ARGS="-DGGML_CUDA=on"`, 15–30분)를 하거나,
- 아래 1절에서 `BACKEND = "hf"`로 바꿔 **transformers로 같은 Phi-3를 돌립니다**(1·2장에서 받은 가중치 재사용, llama.cpp 불필요). 이후 셀은 두 경우 모두 그대로 동작합니다.

In [ ]:
import sys, subprocess, importlib.util
if importlib.util.find_spec("llama_cpp") is None:
    if sys.version_info[:2] <= (3, 12):
        subprocess.call([sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python==0.3.19",
                         "--only-binary=llama-cpp-python",
                         "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu124"])
    else:
        print(f"Python {sys.version_info.major}.{sys.version_info.minor}용 CUDA wheel이 없습니다. 1절에서 BACKEND='hf'를 쓰세요.")
try:
    import llama_cpp
    print("llama-cpp-python", llama_cpp.__version__, "| GPU offload 지원:", llama_cpp.llama_supports_gpu_offload())
except Exception as e:
    print("llama_cpp 불러오기 실패 →", type(e).__name__, str(e)[:200])
    print("libcudart 관련 오류면 BACKEND='hf'로 진행하세요.")

## 1. Model I/O: 양자화 모델 불러오기 (PDF 281–283)

**양자화(quantization)**는 모델 파라미터를 표현하는 비트 수를 줄이는 압축입니다. 책의 비유로는 "14시 16분 12초" 대신 "14시 16분"이라고 말하는 것과 같습니다. 정밀도는 조금 잃지만 메모리와 속도에서 크게 이득을 봅니다. **GGUF**는 llama.cpp가 쓰는 양자화 모델 파일 형식입니다.

책 팁: 4비트 이상을 고르고, 3비트·2비트로 내려야 할 정도면 차라리 더 작은 모델의 높은 정밀도 버전을 쓰는 편이 낫습니다.

**(+ 보충)** 책 그림 7-2(π를 32비트·16비트로 표현)를 숫자로 확인합니다. 비트를 반으로 줄이면 표현 가능한 자릿수가 줄어듭니다.

In [ ]:
import numpy as np
for dt in (np.float64, np.float32, np.float16):
    v = dt(np.pi)
    print(f"{np.dtype(dt).name:8s} {np.dtype(dt).itemsize*8:2d}bit  {float(v):.15f}  오차 {abs(float(v)-np.pi):.2e}")

GGUF 파일을 내려받습니다. 책은 `wget`을 쓰지만 여기서는 `hf_hub_download`를 씁니다. 끊겨도 이어받고, 이미 있으면 다시 받지 않습니다.

> **책과 다를 수 있는 점.** 책 본문은 "8비트 버전을 써서 메모리를 절반으로 줄인다"고 설명하지만, 실제 코드는 `fp16`(16비트, 7.6GB) 파일을 받습니다. Microsoft 저장소에는 `fp16`과 `q4`(4비트, 2.4GB) 두 가지만 있습니다. 8장은 `q4`를 씁니다.

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

BACKEND = "llamacpp"                               # llama.cpp 설치가 안 되면 "hf"
GGUF_FILE = "Phi-3-mini-4k-instruct-fp16.gguf"     # 스토리지가 빠듯하면 "Phi-3-mini-4k-instruct-q4.gguf"
MODEL_DIR = Path.home() / "models"

if BACKEND == "llamacpp":
    model_path = hf_hub_download("microsoft/Phi-3-mini-4k-instruct-gguf", GGUF_FILE, local_dir=MODEL_DIR)
    print(model_path, f"{Path(model_path).stat().st_size/1e9:.1f} GB")

LangChain의 `LlamaCpp` 래퍼로 불러옵니다. `n_gpu_layers=-1`은 모든 층을 GPU에 올리라는 뜻입니다. `BACKEND="hf"`이면 같은 Phi-3를 transformers로 불러 `HuggingFacePipeline`으로 감쌉니다. 어느 쪽이든 `llm.invoke("문자열")` → 문자열 형태는 같습니다.

In [ ]:
if BACKEND == "llamacpp":
    from langchain_community.llms import LlamaCpp     # 책: from langchain import LlamaCpp
    llm = LlamaCpp(
        model_path=model_path,
        n_gpu_layers=-1,
        max_tokens=500,
        n_ctx=2048,
        seed=42,
        verbose=False,
    )
else:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
    from langchain_huggingface import HuggingFacePipeline
    hf_model = AutoModelForCausalLM.from_pretrained(
        "microsoft/Phi-3-mini-4k-instruct", device_map="cuda", torch_dtype="auto",
        trust_remote_code=False,   # 책 1장 본문은 True. 최신 transformers에는 Phi-3가 내장되어 False가 안전
    )
    hf_tok = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
    gen = pipeline("text-generation", model=hf_model, tokenizer=hf_tok,
                   max_new_tokens=500, do_sample=False, return_full_text=False)
    llm = HuggingFacePipeline(pipeline=gen)
print(type(llm).__name__)

**관찰 포인트.** 책에서는 이 호출이 빈 문자열 `''`을 돌려줍니다. Phi-3는 `<|user|> … <|end|> <|assistant|>` 형식의 채팅 템플릿을 기대하는데, transformers의 `pipeline`과 달리 LlamaCpp 래퍼는 템플릿을 자동으로 씌우지 않기 때문입니다. 버전에 따라 빈 문자열 대신 엉뚱한 문장이 나올 수도 있습니다. 이 문제를 다음 절의 템플릿 체인으로 풉니다.

In [ ]:
llm.invoke("Hi! My name is Maarten. What is 1 + 1?")

## 2. Chains: 부품을 잇기 (PDF 284–292)

**체인(chain)**은 LLM에 다른 부품(프롬프트 템플릿, 메모리, 도구 등)을 이어 붙인 실행 단위입니다. 가장 단순한 체인은 "템플릿 → LLM" 한 줄입니다. 템플릿에 `{input_prompt}` 자리를 두면 매번 질문만 바꿔 넣을 수 있습니다.

`prompt | llm`의 `|`는 LangChain 표현 언어(LCEL, LangChain Expression Language)의 연결 연산자입니다. 앞 단계의 출력이 다음 단계의 입력이 됩니다.

In [ ]:
from langchain_core.prompts import PromptTemplate   # 책: from langchain import PromptTemplate

template = """<s><|user|>
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(template=template, input_variables=["input_prompt"])

basic_chain = prompt | llm
basic_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

### 여러 프롬프트를 잇는 체인 (PDF 289–292)

긴 이야기를 한 번에 쓰게 하지 않고 **제목 → 주인공 묘사 → 본문** 세 단계로 나눕니다. 각 단계의 출력(`output_key`)이 다음 단계 템플릿의 변수로 들어갑니다. 작은 모델도 단계를 나누면 각 단계에 집중할 수 있다는 것이 책의 요점입니다.

책 코드의 `LLMChain`은 LangChain 0.1.17부터 사용 중단(deprecated) 예고가 붙었고, 지금은 `langchain_classic`에 있습니다. 실행하면 `LangChainDeprecationWarning`(2.0에서 제거 예정)이 나오지만 동작에는 문제가 없습니다. 3절의 메모리 클래스도 같은 경고가 나옵니다.

In [ ]:
from langchain_classic.chains import LLMChain        # 책: from langchain import LLMChain

template = """<s><|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>"""
title_prompt = PromptTemplate(template=template, input_variables=["summary"])
title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")
title.invoke({"summary": "a girl that lost her mother"})

In [ ]:
template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}. Use only two sentences.<|end|>
<|assistant|>"""
character_prompt = PromptTemplate(template=template, input_variables=["summary", "title"])
character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

template = """<s><|user|>
Create a story about {summary} with the title {title}. The main charachter is: {character}. Only return the story and it cannot be longer than one paragraph<|end|>
<|assistant|>"""
story_prompt = PromptTemplate(template=template, input_variables=["summary", "title", "character"])
story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

llm_chain = title | character | story
llm_chain.invoke("a girl that lost her mother")

**관찰 포인트.** 결과 딕셔너리에 `summary`, `title`, `character`, `story`가 모두 남습니다. `LLMChain`이 입력을 그대로 넘기면서 자기 출력을 덧붙이기 때문에 뒤 단계가 앞 단계의 모든 값을 쓸 수 있습니다.

**(+ 보충) 현재 권장 방식(LCEL).** 같은 체인을 `LLMChain` 없이 씁니다. `RunnablePassthrough.assign(키=체인)`은 "입력 딕셔너리를 그대로 두고 새 키를 하나 더한다"는 뜻으로, `LLMChain`의 `output_key`와 같은 역할입니다. `.strip()`은 모델 출력 앞뒤 공백을 다듬습니다.

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
clean = StrOutputParser() | (lambda s: s.strip())

story_lcel = (
    RunnablePassthrough.assign(title=title_prompt | llm | clean)
    | RunnablePassthrough.assign(character=character_prompt | llm | clean)
    | RunnablePassthrough.assign(story=story_prompt | llm | clean)
)
story_lcel.invoke({"summary": "an autonomous car that learns to drive in the snow"})

## 3. Memory: 대화 기억하기 (PDF 293–305)

LLM은 호출과 호출 사이에 아무것도 기억하지 않습니다(stateless). 아래 두 번째 질문에서 이름을 모르는 것이 그 증거입니다. "기억"은 이전 대화를 **프롬프트에 다시 넣어 주는 것**으로 구현합니다.

In [ ]:
print(basic_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"}))
print(basic_chain.invoke({"input_prompt": "What is my name?"}))

### 3-1. Conversation Buffer: 전체 대화를 그대로 (PDF 295–297)

템플릿에 `{chat_history}` 자리를 만들고, 메모리 객체가 매 호출마다 지금까지의 대화를 채워 넣습니다.

In [ ]:
template = """<s><|user|>Current conversation:{chat_history}

{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(template=template, input_variables=["input_prompt", "chat_history"])

from langchain_classic.memory import ConversationBufferMemory   # 책: from langchain.memory import ...
memory = ConversationBufferMemory(memory_key="chat_history")
llm_chain = LLMChain(prompt=prompt, llm=llm, memory=memory)

llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

In [ ]:
llm_chain.invoke({"input_prompt": "What is my name?"})

### 3-2. Windowed Conversation Buffer: 최근 k턴만 (PDF 297–300)

대화가 길어지면 히스토리가 컨텍스트 창(이 설정에서는 `n_ctx=2048` 토큰)을 넘습니다. 최근 `k`번의 주고받기만 남기면 길이는 일정해지지만 오래된 정보는 잊습니다.

**관찰 포인트.** `k=2`이므로 세 번째 질문 시점에는 첫 대화(이름·나이)가 아직 남아 있어 이름을 맞힙니다. 네 번째 질문에서는 첫 대화가 창 밖으로 밀려나 나이를 모릅니다.

In [ ]:
from langchain_classic.memory import ConversationBufferWindowMemory
memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")
llm_chain = LLMChain(prompt=prompt, llm=llm, memory=memory)

llm_chain.invoke({"input_prompt": "Hi! My name is Maarten and I am 33 years old. What is 1 + 1?"})
llm_chain.invoke({"input_prompt": "What is 3 + 3?"})

In [ ]:
llm_chain.invoke({"input_prompt": "What is my name?"})

In [ ]:
llm_chain.invoke({"input_prompt": "What is my age?"})

### 3-3. Conversation Summary: 요약해서 (PDF 300–305)

대화를 통째로 넣는 대신 **LLM이 대화를 요약한 문장**을 넣습니다. 매 턴마다 요약용 LLM 호출이 한 번 더 일어나서 느리지만, 히스토리 길이가 잘 늘지 않습니다. 요약 과정에서 세부 정보가 빠지거나 왜곡될 수 있다는 점도 결과에서 확인하세요(책 출력에서도 요약에 "개인정보를 보관하지 않는다"는 엉뚱한 문장이 섞였습니다).

In [ ]:
summary_prompt_template = """<s><|user|>Summarize the conversations and update with the new lines.

Current summary:
{summary}

new lines of conversation:
{new_lines}

New summary:<|end|>
<|assistant|>"""
summary_prompt = PromptTemplate(input_variables=["new_lines", "summary"], template=summary_prompt_template)

from langchain_classic.memory import ConversationSummaryMemory
memory = ConversationSummaryMemory(llm=llm, memory_key="chat_history", prompt=summary_prompt)
llm_chain = LLMChain(prompt=prompt, llm=llm, memory=memory)

llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})
llm_chain.invoke({"input_prompt": "What is my name?"})

In [ ]:
llm_chain.invoke({"input_prompt": "What was the first question I asked?"})

In [ ]:
memory.load_memory_variables({})

**책 표 7-1 정리.**

| 메모리 | 장점 | 단점 |
|---|---|---|
| Buffer (전체) | 구현이 쉽고 정보 손실이 없음 | 대화가 길수록 느려지고 컨텍스트 창을 넘음 |
| Windowed (최근 k턴) | 길이가 일정, 큰 컨텍스트 불필요 | k턴 이전 정보는 잊음, 압축 없음 |
| Summary (요약) | 긴 대화도 짧게 유지 | 매 턴 LLM 호출 추가(느림), 요약 품질에 좌우 |

**(+ 보충) 메모리별 토큰 수.** 같은 6턴 대화를 세 메모리에 넣고, 다음 프롬프트에 실려 갈 히스토리가 몇 토큰인지 셉니다(LLM 호출 없이 메모리 객체만 씁니다. 요약은 LLM이 필요해 앞 셀 결과를 씁니다). 토큰 수는 Phi-3 토크나이저로 셉니다.

In [ ]:
from transformers import AutoTokenizer
phi_tok = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
ntok = lambda s: len(phi_tok(s, add_special_tokens=False).input_ids)

turns = [(f"Question {i}: the car ahead braked at intersection {i}. What should I do?",
          f"Answer {i}: keep a safe following distance and slow down before intersection {i}.") for i in range(1, 7)]
buf = ConversationBufferMemory(memory_key="chat_history")
win = ConversationBufferWindowMemory(k=2, memory_key="chat_history")
for q, a in turns:
    for m in (buf, win):
        m.save_context({"input_prompt": q}, {"text": a})

import pandas as pd
pd.DataFrame({
    "memory": ["Buffer", "Window k=2", "Summary (앞 셀, 3턴)"],
    "history_tokens": [ntok(buf.load_memory_variables({})["chat_history"]),
                       ntok(win.load_memory_variables({})["chat_history"]),
                       ntok(memory.load_memory_variables({})["chat_history"])],
})

**(+ 보충) 현재 권장 방식.** LangChain 1.x는 메모리 클래스 대신 "메시지 목록을 직접 관리하고 매번 프롬프트에 넣는" 방식을 권합니다. 원리는 위 Buffer와 같습니다. 창 방식은 목록을 자르면 되고(`history[-2*k:]`), 요약 방식은 요약 문자열을 따로 들고 다니면 됩니다.

In [ ]:
history = []   # (role, text) 목록 = 메모리
def chat(user_text, k=None):
    h = history if k is None else history[-2 * k:]
    hist = "\n".join(f"{r}: {t}" for r, t in h)
    out = basic_chain.invoke({"input_prompt": f"Current conversation:\n{hist}\n\n{user_text}"}).strip()
    history.extend([("Human", user_text), ("AI", out)])
    return out

print(chat("Hi! My name is Maarten. What is 1 + 1?"))
print(chat("What is my name?"))

In [ ]:
import gc
del llm_chain, memory
_ = gc.collect()

## 4. Agents: 도구를 쓰는 LLM (PDF 306–315)

**에이전트(agent)**는 LLM이 다음에 할 행동을 스스로 고르고, 도구(tool)를 실행해 결과를 보고, 다시 판단하는 시스템입니다. 그 뼈대가 **ReAct**(Reasoning and Acting)입니다.

- **Thought:** 지금 무엇을 해야 하는지 생각을 글로 씀
- **Action / Action Input:** 쓸 도구 이름과 입력을 씀
- **Observation:** 프로그램이 도구를 실제로 실행하고 결과를 프롬프트에 붙임
- 이 과정을 반복하다가 **Final Answer**로 끝냄

LLM은 글만 쓸 뿐이고, 도구 실행은 바깥 프로그램(`AgentExecutor`)이 맡습니다. 이 구분이 중요합니다.

### 에이전트용 LLM 고르기

책은 "Phi-3 같은 작은 모델은 ReAct 형식을 잘 못 따른다"며 OpenAI GPT-3.5를 씁니다. 아래 셀은 다음 순서로 고릅니다.

1. `OPENAI_API_KEY` 환경변수가 있으면 OpenAI. 책의 `gpt-3.5-turbo`는 서비스가 종료될 수 있어 모델명을 변수로 뺐습니다.
2. 없으면 **Amazon Bedrock (책 밖)**. Studio 실행 역할(execution role)에 Bedrock 호출 권한(`bedrock:InvokeModel` 계열)이 있고, 콘솔에서 해당 모델 접근이 켜져 있어야 합니다. 미국 밖 리전에서는 `global.` 접두사의 교차 리전 추론 ID를 씁니다.
3. 둘 다 안 되면 지금 불러 둔 Phi-3로 시도합니다. 형식을 자주 어기므로 `handle_parsing_errors=True`와 반복 횟수 제한을 둡니다.

In [ ]:
import os
OPENAI_MODEL = "gpt-4o-mini"                          # 책: gpt-3.5-turbo
BEDROCK_MODEL_ID = "global.amazon.nova-2-lite-v1:0"   # 콘솔의 Bedrock > Model catalog에서 확인
BEDROCK_REGION = os.environ.get("AWS_REGION", "ap-northeast-2")

agent_llm, agent_name = None, None
if os.environ.get("OPENAI_API_KEY"):
    try:
        import importlib.util, subprocess, sys
        if importlib.util.find_spec("langchain_openai") is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "langchain-openai"])
        from langchain_openai import ChatOpenAI
        agent_llm, agent_name = ChatOpenAI(model=OPENAI_MODEL, temperature=0), f"OpenAI {OPENAI_MODEL}"
    except Exception as e:
        print("OpenAI 설정 실패:", e)
if agent_llm is None:
    try:
        import importlib.util, subprocess, sys
        if importlib.util.find_spec("langchain_aws") is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "langchain-aws"])
        from langchain_aws import ChatBedrockConverse
        cand = ChatBedrockConverse(model=BEDROCK_MODEL_ID, region_name=BEDROCK_REGION, temperature=0)
        cand.invoke("ping")                                   # 권한·모델 접근 확인
        agent_llm, agent_name = cand, f"Bedrock {BEDROCK_MODEL_ID}"
    except Exception as e:
        print("Bedrock 사용 불가 →", type(e).__name__, str(e)[:160])
if agent_llm is None:
    agent_llm, agent_name = llm, "local Phi-3 (형식 오류가 잦을 수 있음)"
print("에이전트 LLM:", agent_name)

책의 ReAct 템플릿입니다. `{tools}`에는 도구 설명이, `{agent_scratchpad}`에는 지금까지의 Thought/Action/Observation 기록이 채워집니다.

In [ ]:
react_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""
react_prompt = PromptTemplate(template=react_template,
                              input_variables=["tools", "tool_names", "input", "agent_scratchpad"])

도구 두 개를 준비합니다. DuckDuckGo 웹 검색과 계산기(`llm-math`)입니다. 계산기는 LLM이 수식을 파이썬 식으로 바꾸고 `numexpr`로 계산하는 체인입니다.

웹 검색은 네트워크 차단이나 요청 제한(rate limit)으로 실패할 수 있습니다. 그래서 검색 함수를 감싸서, 실패하면 에러 대신 "검색 실패" 문장을 Observation으로 돌려주게 했습니다. 에이전트가 멈추지 않고 다른 방법을 찾게 하려는 것입니다.

In [ ]:
from langchain_classic.agents import Tool                         # 책: from langchain.agents import load_tools, Tool
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain_community.tools import DuckDuckGoSearchResults     # 책: from langchain.tools import ...

search = DuckDuckGoSearchResults()
def safe_search(q: str) -> str:
    try:
        return search.run(q)[:2000]
    except Exception as e:
        return f"search failed ({type(e).__name__}). Try answering with the calculator or what you know."

search_tool = Tool(name="duckduck",
                   description="A web search engine. Use this to as a search engine for general queries.",
                   func=safe_search)
tools = load_tools(["llm-math"], llm=agent_llm)
tools.append(search_tool)
[t.name for t in tools]

In [ ]:
from langchain_classic.agents import AgentExecutor, create_react_agent

agent = create_react_agent(agent_llm, tools, react_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True,
                               max_iterations=6, return_intermediate_steps=True)

result = agent_executor.invoke({"input": "What is the current price of a MacBook Pro in USD? "
                                         "How much would it cost in EUR if the exchange rate is 0.85 EUR for 1 USD?"})
result["output"]

**관찰 포인트.** 책도 강조하듯, 답이 그럴듯해도 **맞는지는 따로 확인해야 합니다**. 중간 과정에 사람이 끼어들지 않기 때문입니다. 책이 제안하는 보완책은 출처 URL을 함께 돌려주게 하거나 단계마다 확인을 받는 것입니다.

**(+ 보충) 실행 기록 보기.** `return_intermediate_steps=True`로 받은 기록을 표로 펼칩니다. 각 행이 ReAct 한 바퀴(Thought → Action → Observation)입니다.

In [ ]:
rows = []
for act, obs in result["intermediate_steps"]:
    thought = act.log.split("Action:")[0].strip() if hasattr(act, "log") else ""
    rows.append({"thought": thought[:120], "tool": act.tool, "tool_input": str(act.tool_input)[:80],
                 "observation": str(obs)[:120]})
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(rows)

### (+ 보충) ReAct 루프를 직접 짜 보기

`AgentExecutor`가 하는 일은 결국 아래 30줄 정도입니다. **LLM에게 다음 줄을 쓰게 하고 → `Action:`을 정규식으로 뽑아 → 도구를 실행해 → `Observation:`을 붙여 다시 묻기**를 반복합니다. 멈춤 문자열(stop)로 `Observation:` 앞에서 생성을 끊는 것이 핵심입니다. LLM이 관찰 결과를 지어내지 못하게 하려는 장치입니다.

도구로 **정지거리 계산기**를 넣었습니다(책 밖 예시). 공주거리(반응 시간 동안 달린 거리) + 제동거리(v²/2μg)로 계산합니다.

In [ ]:
import re

def stopping_distance(arg: str) -> str:
    """'speed_kmh, friction' 예: '60, 0.7' → 정지거리(m)"""
    nums = [float(x) for x in re.findall(r"[\d.]+", arg)]
    v = nums[0] / 3.6; mu = nums[1] if len(nums) > 1 else 0.7; t_react = 1.0
    return f"{v * t_react + v**2 / (2 * mu * 9.81):.1f} m (reaction 1.0 s, friction {mu})"

my_tools = {"stopping_distance": stopping_distance, "calculator": lambda e: str(eval(e, {"__builtins__": {}}))}
tool_desc = ("stopping_distance: input 'speed_kmh, friction' -> total stopping distance in meters\n"
             "calculator: input a python arithmetic expression")

def mini_react(question, model=agent_llm, max_steps=5):
    scratch = ""
    for step in range(max_steps):
        p = react_template.format(tools=tool_desc, tool_names=", ".join(my_tools), input=question, agent_scratchpad=scratch)
        out = model.bind(stop=["\nObservation"]).invoke(p)
        out = getattr(out, "content", out)                        # 채팅 모델이면 .content
        out = out if isinstance(out, str) else str(out)
        print(f"--- step {step+1} ---\n{out.strip()}")
        if "Final Answer:" in out:
            return out.split("Final Answer:")[-1].strip()
        m = re.search(r"Action:\s*(\w+)\s*Action Input:\s*(.+)", out, re.S)
        if not m:
            scratch += out + "\nObservation: invalid format, use Action/Action Input.\nThought:"; continue
        name, arg = m.group(1), m.group(2).strip().strip('"').splitlines()[0]
        obs = my_tools[name](arg) if name in my_tools else f"unknown tool {name}"
        print(f"Observation: {obs}")
        scratch += out + f"\nObservation: {obs}\nThought:"
    return "(max steps reached)"

mini_react("On a wet road (friction 0.4), how many meters longer is the stopping distance at 80 km/h than on a dry road (friction 0.8)?")

### (+ 보충, 책 밖) 텍스트 ReAct 대신 도구 호출(tool calling) API

책이 나온 뒤 주류가 된 방식입니다. 모델이 `Action: …` 문자열을 쓰고 정규식으로 파싱하는 대신, **모델 API가 "이 함수를 이 인자로 부르라"는 구조화된 데이터(JSON)를 직접 돌려줍니다**. 형식 오류가 크게 줄어듭니다. 아래 셀은 채팅 모델(OpenAI·Bedrock)일 때만 실행됩니다.

여러 도구 서버를 표준 방식으로 연결하는 MCP(Model Context Protocol)도 이 도구 호출 위에 올라간 규약입니다.

In [ ]:
from langchain_core.tools import tool

@tool
def stopping_distance_tool(speed_kmh: float, friction: float = 0.7) -> str:
    """Total stopping distance in meters for a given speed (km/h) and road friction coefficient."""
    return stopping_distance(f"{speed_kmh}, {friction}")

if hasattr(agent_llm, "bind_tools") and agent_llm is not llm:
    msg = agent_llm.bind_tools([stopping_distance_tool]).invoke("Stopping distance at 100 km/h on an icy road (friction 0.1)?")
    print("tool_calls:", msg.tool_calls)
    for c in msg.tool_calls:
        print("→ 실행 결과:", stopping_distance_tool.invoke(c["args"]))
else:
    print("채팅 모델(OpenAI/Bedrock)이 없어 건너뜁니다.")

## 5. 정리

| 부품 | 하는 일 | 이 노트북에서 쓴 것 |
|---|---|---|
| Model I/O | 모델 불러오기·호출 | GGUF + llama.cpp (`LlamaCpp`) 또는 `HuggingFacePipeline` |
| Prompt template | 모델별 입력 형식을 고정하고 변수만 바꿔 넣기 | `PromptTemplate` |
| Chain | 부품을 순서대로 연결 | `prompt \| llm`, `LLMChain`, `RunnablePassthrough.assign` |
| Memory | 이전 대화를 프롬프트에 다시 넣기 | Buffer / Window / Summary |
| Agent | LLM이 도구 사용을 스스로 결정 | ReAct 템플릿 + `AgentExecutor`, 직접 짠 루프 |

**자율주행·Agent로 이어지는 점(책 밖).** Agentic AI의 기본 구조는 이 장의 ReAct 루프(판단 → 도구 실행 → 관찰)이고, 지금은 텍스트 파싱 대신 도구 호출 API와 MCP로 도구를 연결합니다. 자율주행 개발 업무에 쓰는 에이전트라면 도구가 "주행 로그 검색", "시나리오 시뮬레이션 실행", "지표 계산" 같은 것이 됩니다. 이때도 책의 경고, 즉 중간 단계에 사람이 없으니 결과 검증 장치를 설계해야 한다는 점이 그대로 적용됩니다. 메모리의 토큰 비용 문제는 긴 주행 로그를 다루는 에이전트에서 더 커지는데, 8장의 검색(RAG)이 그 해법 중 하나입니다.

**끝나면** Studio에서 space를 **Stop** 하세요. 내려받은 GGUF 파일은 `~/models`에 남아 8장에서 재사용할 수 있습니다(8장은 q4 파일 사용).